This notebook pulls raw data from Snowflake and store in s3, similar to the procedures in `load_data.py`

In [1]:
import os
os.chdir('../')

In [2]:
import json

import typing as t
from datetime import datetime

import numpy as np
import pandas as pd
from s3fs import S3FileSystem

# from pre_90_pd.utilities import config
from dspantry.pre_90_pd.utilities.data_warehouse import DataWarehouseConnection
from dspantry.pre_90_pd.utilities.functions import split_ids_by_range, timestamp_str

In [3]:
## change values in this cell depending on your requirements
run_date_str = '20241204'  # queries will pull data up to 1 day prior to run_date_str
volume_start_dt = '20230101' # pull daily revenue data starting from this date
output_dir = f's3://toast-datascience-sandbox/PradeepA/pre90_pd_model/prod/{run_date_str}' # directory to store the raw data
transaction_output_dir = os.path.join(output_dir, 'daily_data')  # directory to store the daily revenue data parquet files
os.environ["SNOWFLAKE_PRIVATE_KEY_PASSWORD"] = ""

In [4]:
partial_days = 0
# partial_days = 1 This is usually how it is coded in production. It is made zero above because of adhoc runs.
num_days_volume_data = (pd.to_datetime(run_date_str) - pd.to_datetime(volume_start_dt)).days + 1 - partial_days

datetime_now = datetime.strptime(run_date_str, "%Y%m%d")
end_date = timestamp_str("%Y%m%d", days=partial_days, time_now=datetime_now)

ts_start_date = timestamp_str(
    "%Y%m%d",
    days=partial_days + num_days_volume_data - 1,
    time_now=datetime_now
)

In [5]:
ts_start_date, end_date

('20230101', '20241204')

In [6]:
accts = DataWarehouseConnection().get_sf_account_details(ts_start_date, end_date)
accts.to_parquet(os.path.join(output_dir, 'accts.parquet'))

In [7]:
monthly_modules = DataWarehouseConnection().get_live_modules(ts_start_date, end_date)
monthly_modules.to_parquet(os.path.join(output_dir, 'monthly_modules.parquet'))

In [8]:
## daily revenue data
rid_ls = accts['rid'].tolist()
rid_group_dict = split_ids_by_range(rid_ls)
for k, v in rid_group_dict.items():
    print('...loading rid chunk %s', k)
    daily_ = DataWarehouseConnection().get_daily_rev(
        start_yyyymmdd=ts_start_date,
        end_yyyymmdd=end_date,
        rest_ids=list(map(str, v)),
    )
    daily_.rename(columns={'restaurant_id': 'rid'}, inplace=True)
    if len(daily_) > 0:
        print('...saving rid chunk %s', k)
        daily_.to_parquet(os.path.join(transaction_output_dir, f'rid-{k}.parquet'))

...loading rid chunk %s 230000000000000000
...saving rid chunk %s 230000000000000000
...loading rid chunk %s 210000000000000000
...saving rid chunk %s 210000000000000000
...loading rid chunk %s 60000000000000000
...saving rid chunk %s 60000000000000000
...loading rid chunk %s 100000000000000000
...saving rid chunk %s 100000000000000000
...loading rid chunk %s 50000000000000000
...saving rid chunk %s 50000000000000000
...loading rid chunk %s 220000000000000000
...saving rid chunk %s 220000000000000000
...loading rid chunk %s 40000000000000000
...saving rid chunk %s 40000000000000000
...loading rid chunk %s 110000000000000000
...saving rid chunk %s 110000000000000000
...loading rid chunk %s 200000000000000000
...saving rid chunk %s 200000000000000000
...loading rid chunk %s 180000000000000000
...saving rid chunk %s 180000000000000000
...loading rid chunk %s 130000000000000000
...saving rid chunk %s 130000000000000000
...loading rid chunk %s 120000000000000000
...saving rid chunk %s 12000

In [9]:
df_pre90_tc_apps = DataWarehouseConnection().get_pre90_tc_apps(ts_start_date, end_date)
df_pre90_tc_apps.to_parquet(os.path.join(output_dir, 'df_pre90_tc_apps.parquet'))

In [10]:
df_pre90_tc_apps

,entity_identifier,rid,funding_requested_date,term,IS_PRE_90
0,b7ad40ea-0cdf-445c-9b6f-505a876b86b3,185600000000000000,2024-01-09,90,True
1,e49bd418-2d47-4d2a-bb8a-4fe3f2981ac6,134623000000000000,2023-05-09,90,True
2,63676f3e-603f-4fb9-b72b-a6bb87282f7f,154036000000000000,2023-05-10,90,True
3,9233404f-3294-4ab9-8425-f6fac911f5cd,213834000000000000,2024-08-27,90,True
4,b9efaa75-b4ed-486f-8c27-fbd3bbca1445,167348000000000000,2023-09-20,90,True
...,...,...,...,...,...
4772,35bcab2b-504e-42fc-80cc-4143b5375793,124270000000000000,2023-01-28,90,True
4773,72f00dcf-e1b8-4d0e-82ce-676a0224456c,189657000000000000,2024-04-23,90,True
4774,acc025c9-d770-4a5f-919a-a95887f32b38,147455000000000000,2024-05-27,90,True
4775,981610b4-13f1-4806-8785-0c74778b1fee,231616000000000000,2024-11-11,90,True
